In [4]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import glob
sns.set(rc={'axes.facecolor':'skyblue'})
plt.rcParams['figure.dpi'] = 300

In [5]:
def assign_tercile_category(dataframe, variable_name, dataset_name, region_name=''):
  """Takes a dataframe and a variable name, and returns a dataframe with a column
  for the tercile category of the variable.

  Assumptions:
  Desired model has been converted to dataframe format
  You know the variable of interest to compute terciles for, such as 'precip'

  Usage:
  GFDL_tercile_category_df = assign_tercile_category(GFDL_south_sudan_df, 'predicted_precip', 'GFDL')

  The dataset_name argument makes the dataframe more readable and convenient for analysis

  Returns:
  A dataframe with a column for the tercile category of the variable.
  """

  # Compute tercile thresholds
  lower_tercile = dataframe[variable_name].quantile(0.33)
  upper_tercile = dataframe[variable_name].quantile(0.66)

  # Classify precipitation using vectorized NumPy operations
  conditions = [
        dataframe[variable_name] < lower_tercile,
        dataframe[variable_name] > upper_tercile
    ]
  choices = ["Low", "High"]

  # Assign categories (default is "Medium")
  dataframe[f"tercile_category_{dataset_name}"] = np.select(conditions, choices, default="Medium")

  print(f'Lower Tercile Cutoff for {dataset_name} {region_name}: {lower_tercile}')
  print(f'Upper Tercile Cutoff for {dataset_name} {region_name}: {upper_tercile}')

  return dataframe

In [6]:
# a function that generates a tercile dataframe for a given tercile
# tercile category must be 'Low', 'Medium', 'High'
def generate_tercile_dataframe(file_path_merged_netcdf, tercile_category='Low', year_lower=1993, year_upper=2024):

  # extract the model and region from the file path of merged netcdf
  model = file_path_merged_netcdf.split('/')[-1].split('_')[-2]
  region = file_path_merged_netcdf.split('/')[-1].split('_')[0:-2]
  region = '_'.join(region)

  # print the model and region of the curent file
  print(f'Model: {model}')
  print(f'Region: {region}')
  print(f'Tercile Category: {tercile_category}')

  # open the merged netcdf, convert to a dataframe, and reset index
  current_netcdf = xr.open_dataset(file_path_merged_netcdf)
  current_df = current_netcdf.to_dataframe()
  current_df = current_df.reset_index()

  try:
    # subset where year is 1993 to 2020
    current_df = current_df[(current_df['time'].dt.year >= year_lower) & (current_df['time'].dt.year <= year_upper)]

    # drop na values, these are places where chirps is measuring the ocean etc
    current_df = current_df.dropna()

  except Exception as e:
    print(e)

  # take the ensemble mean
  current_df = (current_df
                            .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                            .mean().reset_index())

  # take the spatial means
  current_df = (current_df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

  # assign tercile category for both chirps and gfdl
  current_df = assign_tercile_category(current_df, 'predicted_precip', dataset_name=model, region_name=region)
  current_df = assign_tercile_category(current_df, 'precip', 'chirps', region_name=region) # precip in our case will always come from chirps

  # extract month and year from time value of spatial means
  current_df['month'] = current_df['time'].dt.month
  current_df['year'] = current_df['time'].dt.year

  # drop uneeded time column
  current_df = current_df.drop('time', axis=1)

  # compute agreement by month and lead time (T/F)
  current_df['agreement_rate'] = current_df['tercile_category_chirps'] == current_df[f'tercile_category_{model}']

  # subset where chirps categorized precip as low, medium, or high, based on user input of tercile_category value
  current_df = current_df[current_df['tercile_category_chirps'] == tercile_category]

  # mean by year
  current_df = current_df.groupby(['month', 'lead_time'])['agreement_rate'].mean().reset_index()

  # set the model and category for facet
  current_df['model'] = model
  current_df['region'] = region
  current_df['tercile_analysis'] = tercile_category

  # For all the missing months, assign an agreement rate of na

  # generate all months (1 to 12)
  all_months = pd.DataFrame({"month": range(1, 13)})

  # Merge with the original data to identify missing months
  current_df = all_months.merge(current_df, on="month", how="left")

  # Assign 'na' to missing agreement rates
  current_df["agreement_rate"] = current_df["agreement_rate"].fillna(np.nan)

  # return current_df, this is the one used for plotting the graph
  # this will plot the tercile agreement rate for the average precip of the entire region
  # for each lead time at each month for all years, for one model
  # if chirps never classified a given month of precip as low/medium/high
  # the dataframe will have a nan value for that month
  return current_df

# test function, this can be ran in a for loop for multiple regions
# 3current_df = generate_tercile_dataframe('/content/drive/My Drive/capstone_data/netCDF/south_sudan_NASA_merged.nc', 'Low', 1993, 2024)

In [10]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/netCDF/*CMCC*')

# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = generate_tercile_dataframe(f, 'Low', 1993, 2024)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Now `final_df` contains all concatenated data
# final_df

Model: CMCC
Region: netCDF\eastern_east_africa
Tercile Category: Low
Lower Tercile Cutoff for CMCC netCDF\eastern_east_africa: 0.8852257213713809
Upper Tercile Cutoff for CMCC netCDF\eastern_east_africa: 1.914929572448433
Lower Tercile Cutoff for chirps netCDF\eastern_east_africa: 16.984688529968263
Upper Tercile Cutoff for chirps netCDF\eastern_east_africa: 35.188924102783254
Model: CMCC
Region: netCDF\eastern_ukraine
Tercile Category: Low
Lower Tercile Cutoff for CMCC netCDF\eastern_ukraine: 1.363207454291238
Upper Tercile Cutoff for CMCC netCDF\eastern_ukraine: 1.6755659958417246
Lower Tercile Cutoff for chirps netCDF\eastern_ukraine: 36.19578170776367
Upper Tercile Cutoff for chirps netCDF\eastern_ukraine: 54.43489456176758
Model: CMCC
Region: netCDF\lake_victoria_basin
Tercile Category: Low
Lower Tercile Cutoff for CMCC netCDF\lake_victoria_basin: 3.819068356509132
Upper Tercile Cutoff for CMCC netCDF\lake_victoria_basin: 5.665387761548914
Lower Tercile Cutoff for chirps netCDF\la

In [11]:
# Clean the data
df_clean = final_df.dropna(subset=['model', 'region']).copy()

# Get unique models and regions
models = df_clean['model'].unique()
regions = df_clean['region'].unique()

# Create all combinations of model, region, month, and lead_time
months = range(1, 13)
lead_times = np.arange(0.5, 12, 1)  # 0.5 to 11.5 in steps of 1

# Generate a DataFrame with all possible combinations
all_combinations = pd.MultiIndex.from_product(
    [models, regions, months, lead_times],
    names=['model', 'region', 'month', 'lead_time']
).to_frame(index=False)

# Merge with the cleaned data to fill in agreement_rate
merged_df = all_combinations.merge(
    df_clean[['model', 'region', 'month', 'lead_time', 'agreement_rate']],
    on=['model', 'region', 'month', 'lead_time'],
    how='left'
)

# Convert integer lead times to 0.5 increments (e.g., 1 -> 1.5)
df_clean['lead_time'] = df_clean['lead_time'].apply(
    lambda x: x + 0.5 if x % 1 == 0 else x
)

# Generate grid parameters
lead_times = np.arange(0.5, 12, 1)  # Fixed grid from 0.5-11.5 in 1.0 steps

# Create FacetGrid and plot as before
g = sns.FacetGrid(
    merged_df,
    row='model',
    col='region',
    height=4,
    aspect=2,
    sharex=False,
    sharey=False
)

def draw_heatmap(data, **kwargs):
    pivot_data = data.pivot(index='month', columns='lead_time', values='agreement_rate')
    pivot_data = pivot_data.reindex(index=months, columns=lead_times)

    sns.heatmap(
        pivot_data,
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        annot=False,
        cbar=False,
        square=True,
        linewidths=0.5,
        linecolor='black'
    )
    plt.gca().invert_yaxis()
    plt.xticks(ticks=np.arange(len(lead_times)) + 0.5, labels=lead_times, rotation=90)
    plt.yticks(ticks=np.arange(len(months)) + 0.5, labels=months, rotation=0)

g.map_dataframe(draw_heatmap)
g.set_axis_labels('Lead Time', 'Month')
g.set_titles(col_template="{col_name}", row_template="{row_name}")

# Add colorbar
cbar_ax = g.fig.add_axes([0.92, 0.3, 0.02, 0.4])
sm = plt.cm.ScalarMappable(cmap='RdYlGn', norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
g.fig.colorbar(sm, cax=cbar_ax, label='Agreement Rate')

plt.tight_layout()
plt.savefig('figures/low_tercile-hit_rate.png')
plt.close()

C:\Users\Johnson Leung\AppData\Local\Temp\ipykernel_35164\1113181831.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
